# WoodelfHD, m=100 background: PLTreeSHAP (notebook 2 of 2)

Runs the PLTreeSHAP part of `benchmarks/woodelfhd_m100_experiment.py`.
`pltreeshap` only supports Python 3.8, so the experiment runs in a Python 3.8 micromamba environment made by your
`run_python38_scripts.bash` helper, the same setup as `05_run_pltreeshap_fasttreeshap.ipynb`.

**Runtime:** CPU with High-RAM, in its own Colab session, at the same time as notebook 1.
Estimated compute is about 12–14 h. The two HIGGS BG SHAP cells are single long runs (~3.6 h at D=6, ~5.3 h at D=9).
A disconnect during one of them loses that cell.

**Start this after notebook 1's step 5 (Prepare datasets) has finished once.** Then the datasets are copied from Drive.
Otherwise this run downloads them itself (HIGGS ~20 min) and does not keep them on Drive.

### If the runtime crashes or disconnects
Reconnect and **run all cells from the top**:
- **Results** are saved to Drive after every finished cell (`results/pltreeshap_fasttreeshap.json`). Step 5 restores them and the run skips those cells.
- **Models** are trained straight into a Drive folder (`models_plt/`), so they are not retrained.
- The Python 3.8 environment is set up again by the helper.

Only the cell that was running when the runtime died is lost.
The report is built in notebook 1 (last section) once both notebooks have finished.

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configuration ───────────────────────────────────────────────────
import os, pathlib

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_m100')
METHOD     = 'pltreeshap_fasttreeshap'

# Your Python 3.8 helper (the same file notebook 05 uses)
BASH_SCRIPT_DRIVE_PATH = '/content/drive/MyDrive/ShapResearch/HighDepth/AAAI27/PLTreeSHAP/run_python38_scripts.bash'

REPO_URL    = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'
REPO_BRANCH = 'main'
REPO        = pathlib.Path('/content/treebranchmarks')
TOOLS       = REPO / 'colab' / 'woodelfhd_m100_tools.py'   # restore / save-datasets / progress / report

RESULTS_DRIVE = DRIVE_ROOT / 'results'
RESULTS_DRIVE.mkdir(parents=True, exist_ok=True)
print('Drive folder:', DRIVE_ROOT)

In [ ]:
# ── Step 3: Prepare the Python 3.8 helper ───────────────────────────────────
import shutil
shutil.copy(BASH_SCRIPT_DRIVE_PATH, '/content/run_python38_scripts.bash')
!sed -i 's/\r$//' /content/run_python38_scripts.bash
!chmod +x /content/run_python38_scripts.bash
print('Helper ready.')

In [ ]:
# ── Step 4: Clone the repo ──────────────────────────────────────────────────
# Nothing is installed in this kernel. The helper installs everything into the Python 3.8 environment in step 7.

if not REPO.exists():
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO}
if not (REPO / 'benchmarks' / 'woodelfhd_m100_experiment.py').exists() or not TOOLS.exists():
    raise FileNotFoundError(
        'The m=100 experiment files are not in the cloned repo. Push them to GitHub '
        '(or set REPO_BRANCH), delete /content/treebranchmarks and rerun this cell.')

In [ ]:
# ── Step 5: Restore datasets, models and finished results from Drive ────────
!python {TOOLS} restore --drive-root "{DRIVE_ROOT}" --methods {METHOD} --models-dir models_plt

In [ ]:
# ── Step 6: Progress so far ─────────────────────────────────────────────────
# Shows only the number of finished cells: the experiment can't be imported in this kernel.
# Notebook 1's progress step lists the missing cells for every method.
!python {TOOLS} progress --drive-root "{DRIVE_ROOT}" --methods {METHOD}

## Run

In [ ]:
# ── Step 7: Run the PLTreeSHAP part under Python 3.8 ────────────────────────
# Same package groups as notebook 05.
# --result_location is a folder: the run rewrites pltreeshap_fasttreeshap.json there after every finished cell.
!/content/run_python38_scripts.bash \
    --cwd {REPO} \
    --pip "numpy==1.24.0 scipy pyarrow fastparquet pandas xgboost scikit-learn tqdm" \
    --pip "Cython lightgbm" \
    --pip "woodelf_explainer" \
    --pip "-e {REPO}" \
    --pip "fasttreeshap" \
    --pip "-v --no-build-isolation git+https://github.com/schufa-innovationlab/pltreeshap@main" \
    --run "-m benchmarks.woodelfhd_m100_experiment --method {METHOD} --result_location {RESULTS_DRIVE}"

In [ ]:
# ── Step 8: Progress after the run ──────────────────────────────────────────
!python {TOOLS} progress --drive-root "{DRIVE_ROOT}" --methods {METHOD}